# Gold Feature Engineering

## Context

This notebook transforms `fraud.gold.fraud_dataset_selected` (590,540 rows × 101 cols:
72 numeric + 26 categorical + 3 metadata) into an **enriched feature table** ready for
modeling. No encoding, no imputation, no scaling: those belong to the fold-safe modeling
pipeline. Gold only produces **raw + engineered features**.

**Scope**: all transformations are written in **pure SQL** (ANSI-compatible) for
portability. The notebook runs on Databricks but the SQL is reusable on Snowflake,
BigQuery, PostgreSQL, Redshift with minimal changes. Only the final modeling step
(LightGBM training) will use PySpark.

## Objectives

1. **Build the customer UID** — `card1 + addr1 + D1n`, where `D1n = (transaction_dt / 86400) − D1`.
   `D1n` represents the card creation day and is stable per customer. This is the
   winning strategy of the IEEE-CIS competition: the real challenge is predicting
   *customer* fraud, not *transaction* fraud (68% of the private test set involves
   customers never seen in train).
2. **Create UID aggregations** — 8 features capturing customer-level behavior.
3. **Create velocity features** — 6 features capturing transaction bursts over
   time windows (1h, 24h, 7d).
4. **Create derived time features** — hour of day, day of week, weekend flag.
5. **Create `is_null_*` flags** — 5 features where the absence of a value has been
   empirically proven to be a stronger signal than the value itself.
6. **Extract device family** — reduce `device_os` granularity to 7 families.
7. **Rename known columns** — for SHAP readability.

## Critical methodological decisions

### Decision 1 — Backward-only window functions (no temporal leakage)

**All** window functions use `RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING`
(aggregations) or `RANGE BETWEEN <window> PRECEDING AND 1 PRECEDING` (velocity).
This means each feature only looks at **strictly past** transactions for the same UID.

**Consequence**: the first transaction of every UID gets `NULL` for all UID
aggregations and velocity features. This is correct behavior — in production, the
first transaction of a new customer has no history. LightGBM handles NULL natively.

**Why not full-dataset aggregations**: they would introduce temporal leakage (a
transaction would "see" future transactions of the same customer), inflating offline
metrics by an unknown amount. Backward-only is the honest approach.

### Decision 2 — Feature renaming for SHAP

Known features are renamed to business-readable names. Anonymous Vesta features
(`v258`, `c13`, `d5`, `m4`, `id_15`) are **kept as-is** — any renaming would be
fabrication. A `feature_catalog` documents their family and hypothesis for the README.

Mapping is applied **only at the end** (before the `CREATE TABLE`), on top of the
enriched dataset.

### Decision 3 — No train/valid/test split in this notebook

The temporal split is **not** performed here. It belongs to the modeling pipeline
(or VS Code production code), not to the feature engineering layer. Gold produces a
single enriched table; the split happens when the model needs it.

**Rationale**: split policy (cutoffs, skip periods) may evolve during modeling
experiments. Keeping Gold split-agnostic avoids re-running the notebook for every
split change.

## Pipeline

| Step | Cell | Output |
|---|---|---|
| 05.1 | Column verification | Confirm all 19 rename targets exist |
| 05.2 | UID + D1n construction | `uid`, `d1n`, `uid_is_known` |
| 05.3 | Column renaming | Applied to known features |
| 05.4 | UID aggregations (8 features) | Backward-only |
| 05.5 | Velocity features (6 features) | 1h / 24h / 7d windows |
| 05.6 | Time + flags + device (9 features) | Derived features |
| 05.7 | Materialization | `fraud.gold.fraud_features_enriched` |
| 05.8 | Verification | Row count, null checks, schema |

## Engineered feature inventory

### UID-derived (3)
| Feature | Formula | Purpose |
|---|---|---|
| `d1n` | `(transaction_dt / 86400) − d1` | Card creation day (stable per customer) |
| `uid` | `card1 + addr1 + d1n` | Customer identifier |
| `uid_is_known` | `card1 IS NOT NULL AND addr1 IS NOT NULL` | UID quality flag |

### UID aggregations (8, backward-only)
`uid_transaction_count`, `uid_amount_mean`, `uid_amount_std`, `uid_amount_max`,
`uid_amount_sum`, `uid_distinct_merchants`, `uid_d10_mean`, `uid_d15_mean`

### Velocity (6, backward-only)
`velocity_count_1h`, `velocity_count_24h`, `velocity_count_7d`,
`velocity_amount_sum_24h`, `velocity_amount_sum_7d`,
`velocity_distinct_merchants_24h`

### Time (3)
`hour_of_day`, `day_of_week`, `is_weekend`

### Null flags (5)
`is_null_dist1`, `is_null_dist2`, `is_null_addr1`, `is_null_m6`,
`is_null_device_type`

### Device (1)
`device_family` ∈ {`windows`, `ios`, `macos`, `android`, `linux`, `trident`,
`other`, `missing`}

**Total added**: 26 features. Gold enriched table: 101 + 26 = **127 columns**.

## What Gold does NOT do

| Operation | Where it belongs |
|---|---|
| Target encoding | Modeling pipeline (fold-safe) |
| One-hot encoding | Modeling pipeline (fold-safe) |
| Frequency encoding | Modeling pipeline (fold-safe) |
| Standardization | Modeling pipeline (for linear models only) |
| Imputation | Nowhere — LightGBM handles NULL |
| Train/valid/test split | Modeling pipeline |
| Feature selection (SHAP) | Model evaluation notebook |

## Key insight — `transaction_amt` has zero univariate signal

`transaction_amt` has a univariate AUC of **0.4975** — essentially noise. Its
predictive value comes **exclusively** from aggregations (`uid_amount_mean`,
`uid_amount_std`) and ratios (`amount / uid_amount_mean`) created here. This is
the core finding that distinguishes a serious IEEE-CIS pipeline from a naive one.

## Next step

The enriched table `fraud.gold.fraud_features_enriched` feeds:
- `06_baseline_model` — LightGBM training, AUC-PR measurement
- Production pipeline (VS Code) — Pandera validation, config-driven feature build

## 1-Columns Verification for renaming

In [0]:
%sql

-- Firstly we verify that the columns we want to rename are present
SELECT column_name
FROM fraud.information_schema.columns
WHERE table_schema = 'gold' AND table_name = 'fraud_dataset_selected'
  AND column_name IN ('transaction_amt', 'product_cd', 'card1', 'card2', 'card3',
                      'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1',
                      'dist2', 'p_emaildomain', 'r_emaildomain', 'device_type',
                      'device_info', 'id_31', 'id_33', 'transaction_dt');

column_name
transaction_dt
transaction_amt
dist1
dist2
device_info
id_31
r_emaildomain
card3
product_cd
addr2


In [0]:
%sql
-- Renaming columns using a temprorary view: gold_base_renamed
CREATE OR REPLACE TEMP VIEW gold_base_renamed AS
SELECT
  -- metadata (no changes)
  transaction_id,
  is_fraud,
  transaction_dt,
  -- renaming
  transaction_amt AS amount,
  product_cd AS product_type,
  card1 AS card_id,
  card2 AS card_bank_1,
  card3 AS card_bank_2,
  card4 AS card_network,
  card5 AS card_phone_country,
  card6 AS card_category,
  addr1 AS buyer_region,
  addr2 AS buyer_country_code,
  dist1 AS distance_buyer_merchant,
  dist2 AS distance_secondary,
  p_emaildomain AS buyer_email_domain,
  r_emaildomain AS recipient_email_domain,
  device_info AS device_os,
  id_31 AS browser,
  id_33 AS screen_resolution,
  -- all other terms remain unchanged (V*, C*, D*, M*, id_*, etc.)
  * EXCEPT (
    transaction_id, is_fraud, transaction_dt,
    transaction_amt, product_cd, card1, card2, card3, card4, card5, card6,
    addr1, addr2, dist1, dist2, p_emaildomain, r_emaildomain,
    device_info, id_31, id_33
  )
FROM fraud.gold.fraud_dataset_selected;

### 2. User ID construction

**Why**: On IEEE-CIS, fraud is a *customer* property, not a *transaction* property —
when a card is charged back, the whole account is flagged. The real challenge is
generalizing to **new customers** (68 % of the private test set). To capture
customer-level behavior, we need a stable per-customer identifier. `card1` alone
is insufficient (shared by multiple cards), and `D1` is a per-transaction timedelta
that changes over time.

**How**: We reconstruct the **card creation day** by removing the transaction's
elapsed time (`D1`, in days) from the dataset's chronological index
(`transaction_dt / 86400`, in days):
`d1n` is **stable per card** — two transactions from the same card, at different
dates, produce the same `d1n` (both `transaction_dt` and `d1` advance together).
The `FLOOR` is critical: without it, fractional-day variation fragments each
customer into dozens of pseudo-UIDs.

**Handling missing values**: `COALESCE` maps NULL `card1 → 'unknown'`,
`addr1 → 'no_addr'`, `d1n → 'unknown_day'`. A `uid_is_known` binary flag
records whether both `card1` and `addr1` were present.


In [0]:
%sql
-- Creating User ID feature using a temporary view: v02_with_uid
CREATE OR REPLACE TEMP VIEW v02_with_uid AS
WITH base AS (
  SELECT *,
    CAST(FLOOR((transaction_dt / 86400) - d1) AS BIGINT) AS d1n
  FROM gold_base_renamed
)
SELECT *,
  CONCAT(
    COALESCE(CAST(card_id AS STRING), 'unknown'),
    '_',
    COALESCE(CAST(buyer_region AS STRING), 'no_addr'),
    '_',
    COALESCE(CAST(d1n AS STRING), 'unknown_day')
  ) AS uid,
  CASE WHEN card_id IS NOT NULL AND buyer_region IS NOT NULL THEN 1 ELSE 0 END AS uid_is_known
FROM base;

In [0]:
%sql
WITH stats AS (
  SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT uid) AS distinct_uids,
    SUM(uid_is_known) AS rows_known,
    SUM(1 - uid_is_known) AS rows_unknown,
    COUNT(DISTINCT CASE WHEN uid_is_known = 1 THEN uid END) AS distinct_uids_known,
    COUNT(DISTINCT CASE WHEN uid_is_known = 0 THEN uid END) AS distinct_uids_unknown
  FROM v02_with_uid
)
SELECT
  total_rows,
  distinct_uids,
  ROUND(1.0 * total_rows / distinct_uids, 2) AS avg_transactions_per_uid,
  rows_known,
  ROUND(100.0 * rows_known / total_rows, 2)   AS pct_rows_known,
  rows_unknown,
  ROUND(100.0 * rows_unknown / total_rows, 2) AS pct_rows_unknown,
  distinct_uids_known,
  distinct_uids_unknown
FROM stats;

total_rows,distinct_uids,avg_transactions_per_uid,rows_known,pct_rows_known,rows_unknown,pct_rows_unknown,distinct_uids_known,distinct_uids_unknown
590540,217850,2.71,524834,88.87,65706,11.13,199833,18017


In [0]:
%sql
--Fraud rate by known uid
SELECT
  uid_is_known,
  COUNT(*) AS n,
  SUM(is_fraud) AS frauds,
  ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraud
FROM v02_with_uid
GROUP BY uid_is_known
ORDER BY uid_is_known;

uid_is_known,n,frauds,pct_fraud
0,65706,7741,11.781
1,524834,12922,2.462


**Fraud rate by UID quality** — key finding:

| `uid_is_known` | n | frauds | pct_fraud | vs base (3.499 %) |
|---|---|---|---|---|
| 1 (complete identity) | 524,834 | 12,922 | **2.462 %** | −30 % |
| 0 (incomplete identity) | 65,706 | 7,741 | **11.781 %** | **×3.37** |

**Critical finding — `uid_is_known` is a top-tier signal**:
- Transactions with an **incomplete identity** (missing `card1` or `addr1`)
  represent **11.13 % of the volume** but concentrate **37.5 % of all frauds**
  (7,741 / 20,663).
- Their fraud rate is **4.8× higher** than fully-identified transactions
  (11.78 % vs 2.46 %).

## 3. Aggregations per UID

**Why**: The UID  gives us a stable per-customer identifier, but a
grouping key alone is useless — it must be **exploited** to extract behavioral
signals. The winning insight of IEEE-CIS is that fraud is a *customer* property:
a fraudulent customer's *entire history* is suspect, not just the current
transaction. To capture this, we compute **historical aggregates per UID** —
spending average, variability, volume, activity count. A transaction that
deviates from its own customer's past profile is a stronger fraud signal than
the transaction viewed in isolation.

**How**: 7 aggregations computed with **backward-only windows**
(`RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING`) on `transaction_dt`.
Each feature sees **only strictly past transactions** for the same UID — no
temporal leakage. The first transaction of every customer gets NULL (or 0 for
`COUNT`), which is realistic: a new customer has no history to aggregate.

| Feature | Backward-only | First-transaction value |
|---|---|---|
| `uid_transaction_count` | ✅ | `0` (SQL standard for `COUNT` on empty frame) |
| `uid_amount_mean` | ✅ | NULL |
| `uid_amount_std` | ✅ | NULL (needs 2 points) |
| `uid_amount_max` | ✅ | NULL |
| `uid_amount_sum` | ✅ | NULL |
| `uid_d10_mean` | ✅ | NULL |
| `uid_d15_mean` | ✅ | NULL |

**Dropped**: `uid_distinct_merchants` (`COUNT(DISTINCT) OVER (... RANGE ...)`
unsupported in Spark SQL). Zero exceptions to backward-only — if a feature
cannot be computed cleanly with strictly-past windows, it is dropped, not
special-cased.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v03_with_aggs AS
SELECT *,
  COUNT(*) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_transaction_count,

  AVG(amount) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_amount_mean,

  STDDEV(amount) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_amount_std,

  MAX(amount) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_amount_max,

  SUM(amount) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_amount_sum,

  AVG(d10) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_d10_mean,

  AVG(d15) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
  ) AS uid_d15_mean

FROM v02_with_uid;

In [0]:
%sql
-- Verification
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN uid_transaction_count IS NULL THEN 1 ELSE 0 END) AS null_count,
  SUM(CASE WHEN uid_amount_mean IS NULL THEN 1 ELSE 0 END)   AS null_mean,
  SUM(CASE WHEN uid_amount_std IS NULL THEN 1 ELSE 0 END)   AS null_std,
  SUM(CASE WHEN uid_amount_max IS NULL THEN 1 ELSE 0 END)   AS null_max,
  SUM(CASE WHEN uid_amount_sum IS NULL THEN 1 ELSE 0 END)   AS null_sum,
  SUM(CASE WHEN uid_d10_mean IS NULL THEN 1 ELSE 0 END)   AS null_d10,
  SUM(CASE WHEN uid_d15_mean IS NULL THEN 1 ELSE 0 END)   AS null_d15
FROM v03_with_aggs;

total,null_count,null_mean,null_std,null_max,null_sum,null_d10,null_d15
590540,0,217860,310485,217860,217860,240996,244129


In [0]:

%sql
-- Verify that aggregates are built incrementally on an active UID
SELECT transaction_dt, amount,
  uid_transaction_count, uid_amount_mean, uid_amount_sum, uid_amount_std
FROM v03_with_aggs
WHERE uid = (SELECT uid FROM v03_with_aggs
             GROUP BY uid ORDER BY COUNT(*) DESC LIMIT 1)
ORDER BY transaction_dt
LIMIT 8;

transaction_dt,amount,uid_transaction_count,uid_amount_mean,uid_amount_sum,uid_amount_std
11204367,100.0,0,null,null,null
12081968,10.0,1,100.0,100.0,null
12426792,55.0,2,55.0,110.0,63.63961030678928
12577656,75.0,3,55.0,165.0,45.0
12673229,65.0,4,60.0,240.0,38.07886552931954
12687363,55.0,5,61.0,305.0,33.05298776207682
12750993,75.0,6,60.0,360.0,29.664793948382652
12766647,55.0,7,62.142857142857146,435.0,27.66724038418343


## 4.Velocity features (5 features)

**Method**: 5 features capturing **transaction bursts** over time windows, all
backward-only (`RANGE BETWEEN <window> PRECEDING AND 1 PRECEDING`).

| Feature | Window | Captures |
|---|---|---|
| `velocity_count_1h` | 3,600 s | Instant burst (carding signature) |
| `velocity_count_24h` | 86,400 s | Recent activity |
| `velocity_count_7d` | 604,800 s | Weekly volume |
| `velocity_amount_sum_24h` | 86,400 s | Recent monetary volume |
| `velocity_amount_sum_7d` | 604,800 s | Weekly monetary volume |


**Verification results**:

| Metric | Value |
|---|---|
| Total rows | 590,540 |
| Rows with `velocity_count_1h > 0` | **93,366** (15.8 %) |
| Rows with `velocity_count_24h > 0` | **170,969** (28.9 %) |
| Rows with `velocity_count_7d > 0` | **242,639** (41.1 %) |
| Max `velocity_count_1h` | **191** |
| Max `velocity_count_24h` | **880** |
| Max `velocity_count_7d` | **975** |
| Max `velocity_amount_sum_24h` | **94,901 $** |
| Max `velocity_amount_sum_7d` | **103,440 $** |

**Sanity check — carding signature**:

| Bucket | n | frauds | pct_fraud | vs base (3.5 %) |
|---|---|---|---|---|
| active (1-5 in 1h) | 89,413 | 5,974 | **6.68 %** | ×1.91 |
| burst (>5 in 1h) | 3,953 | 252 | **6.38 %** | ×1.82 |
| no recent activity | 497,174 | 14,437 | **2.90 %** | ×0.83 |




In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v04_with_velocity AS
SELECT *,
  -- Counts Based on Time Windows
  COUNT(*) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
  ) AS velocity_count_1h,

  COUNT(*) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN 86400 PRECEDING AND 1 PRECEDING
  ) AS velocity_count_24h,

  COUNT(*) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN 604800 PRECEDING AND 1 PRECEDING
  ) AS velocity_count_7d,

  -- Totals of Amounts on WindowsS
  SUM(amount) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN 86400 PRECEDING AND 1 PRECEDING
  ) AS velocity_amount_sum_24h,

  SUM(amount) OVER (
    PARTITION BY uid ORDER BY transaction_dt
    RANGE BETWEEN 604800 PRECEDING AND 1 PRECEDING
  ) AS velocity_amount_sum_7d

FROM v03_with_aggs;

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN velocity_count_1h  > 0 THEN 1 ELSE 0 END) AS rows_with_1h,
  SUM(CASE WHEN velocity_count_24h > 0 THEN 1 ELSE 0 END) AS rows_with_24h,
  SUM(CASE WHEN velocity_count_7d  > 0 THEN 1 ELSE 0 END) AS rows_with_7d,
  MAX(velocity_count_1h) AS max_1h,
  MAX(velocity_count_24h) AS max_24h,
  MAX(velocity_count_7d) AS max_7d,
  ROUND(MAX(velocity_amount_sum_24h), 2) AS max_amt_24h,
  ROUND(MAX(velocity_amount_sum_7d), 2)  AS max_amt_7d
FROM v04_with_velocity;

total,rows_with_1h,rows_with_24h,rows_with_7d,max_1h,max_24h,max_7d,max_amt_24h,max_amt_7d
590540,93366,170969,242639,191,880,975,94901.0,103440.0


In [0]:
%sql
SELECT
  CASE
    WHEN velocity_count_1h > 5 THEN 'burst (>5 in 1h)'
    WHEN velocity_count_1h > 0 THEN 'active (1-5 in 1h)'
    ELSE 'no recent activity'
  END AS bucket,
  COUNT(*) AS n,
  SUM(is_fraud) AS frauds,
  ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraud
FROM v04_with_velocity
GROUP BY bucket
ORDER BY pct_fraud DESC;

bucket,n,frauds,pct_fraud
active (1-5 in 1h),89413,5974,6.681
burst (>5 in 1h),3953,252,6.375
no recent activity,497174,14437,2.904


In [0]:
%sql
-- UIDs with extreme max_1h values: merchants or artifacts?
SELECT
  uid, uid_is_known, COUNT(*) AS total_tx, MAX(velocity_count_1h) AS max_1h,
  SUM(is_fraud) AS frauds, ROUND(100.0 * SUM(is_fraud) / COUNT(*), 2) AS pct_fraud
FROM v04_with_velocity
GROUP BY uid, uid_is_known
ORDER BY MAX(velocity_count_1h) DESC
LIMIT 10;

uid,uid_is_known,total_tx,max_1h,frauds,pct_fraud
15775_330.0_129,1,1414,191,0,0.00
9917_no_addr_117,0,76,24,64,84.21
15885_no_addr_20,0,134,21,4,2.99
14276_no_addr_56,0,31,21,25,80.65
15885_no_addr_27,0,137,20,8,5.84
9633_no_addr_21,0,82,19,8,9.76
2616_441.0_35,1,21,19,0,0.00
15885_no_addr_29,0,132,18,2,1.52
15885_no_addr_165,0,78,18,0,0.00
16136_no_addr_144,0,58,18,0,0.00


## 5. Temporal features

Why: raw `transaction_dt` is a monotonically increasing counter — feeding it directly would teach the model "later = more fraud", which is a temporal drift artifact, not a generalizable signal. Instead, we extract **cyclical** temporal features that are periodic and stable: hour of day, day of week, weekend flag.

How: modulo arithmetic on `transaction_dt`:
- hour_of_day = FLOOR((transaction_dt % 86400) / 3600) → 0-23
- day_of_week = FLOOR(transaction_dt / 86400) % 7 → 0-6
- is_weekend = day_of_week IN (5,6) → 0/1

**Findings**:
- Fraud peaks between **7h–9h** (morning hours).
- Weekdays slightly riskier than weekends (**3.53 % vs 3.37 %**).
- Within weekdays, **day 0 and day 3** show elevated fraud rates.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v05_with_time AS
SELECT *,
  FLOOR((transaction_dt % 86400) / 3600) AS hour_of_day,
  FLOOR(transaction_dt / 86400) % 7 AS day_of_week,
  CASE WHEN FLOOR(transaction_dt / 86400) % 7 IN (5, 6) THEN 1 ELSE 0 END AS is_weekend
FROM v04_with_velocity;

In [0]:
%sql
-- Distribution + fraud rate by hour
SELECT hour_of_day, COUNT(*) AS n, SUM(is_fraud) AS frauds,
  ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraud
FROM v05_with_time
GROUP BY hour_of_day
ORDER BY hour_of_day;


hour_of_day,n,frauds,pct_fraud
0,37795,1186,3.138
1,32797,1027,3.131
2,26732,1002,3.748
3,20802,797,3.831
4,14839,770,5.189
5,9701,682,7.030
6,6007,467,7.774
7,3704,393,10.610
8,2591,241,9.301
9,2479,223,8.996


In [0]:
%sql
-- Distribution + fraud rate by day of week
SELECT day_of_week, COUNT(*) AS n, SUM(is_fraud) AS frauds,
  ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraud
FROM v05_with_time
GROUP BY day_of_week
ORDER BY day_of_week;

day_of_week,n,frauds,pct_fraud
0,86377,3211,3.717
1,98502,3550,3.604
2,79834,2963,3.711
3,70223,2503,3.564
4,85433,2687,3.145
5,84815,2803,3.305
6,85356,2946,3.451


In [0]:
%sql
-- Weekend vs weekday
SELECT is_weekend, COUNT(*) AS n, ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraude
FROM v05_with_time
GROUP BY is_weekend;

is_weekend,n,pct_fraude
0,420369,3.548
1,170171,3.378


## 6- Device family extraction

**Why**: `device_os` is high-cardinality (~1,780 distinct values like
`Windows 10`, `iOS 11.2.1`, `Mac OS X 10_12_6`). Feeding the raw string to
the model means target/frequency encoding on thousands of categories — noisy
and prone to overfitting. We reduce it to **7 families** capturing the
essential platform signal (established finding: fraud rates vary strongly by
OS family — see §8bis device drift analysis).

**How**: `CASE WHEN ... LIKE` pattern matching on `device_os`:

| Family | Pattern |
|---|---|
| `windows` | `%windows%` |
| `ios` | `%ios%` |
| `macos` | `%mac%` |
| `android` | `%android%` |
| `linux` | `%linux%` |
| `trident` | `%trident%` (Internet Explorer engine) |
| `missing` | `device_os IS NULL` |
| `other` | everything else |

| Family | n | frauds | pct_fraud | vs base (3.5 %) |
|---|---|---|---|---|
| **other** | 30,893 | 3,859 | **12.49 %** | **×3.57**  |
| windows | 47,775 | 3,131 | 6.55 % | ×1.87 |
| ios | 19,782 | 1,240 | 6.27 % | ×1.79 |
| android | 82 | 3 | 3.66 % | ×1.05 |
| missing | 471,874 | 12,056 | 2.56 % | ×0.73 |
| macos | 12,573 | 278 | 2.21 % | ×0.63 |
| trident | 7,440 | 96 | 1.29 % | ×0.37 |
| linux | 121 | 0 | 0.00 % | — |

**Not created — `device_family_risk`**: the fraud rate per family would be a
target-encoded feature. That belongs to the **fold-safe modeling pipeline**, not
Gold — computing it here on the full dataset would leak the target.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v06_with_device AS
SELECT *,
  CASE
    WHEN device_os IS NULL THEN 'missing'
    WHEN LOWER(device_os) LIKE '%windows%' THEN 'windows'
    WHEN LOWER(device_os) LIKE '%ios%' THEN 'ios'
    WHEN LOWER(device_os) LIKE '%mac%' THEN 'macos'
    WHEN LOWER(device_os) LIKE '%android%' THEN 'android'
    WHEN LOWER(device_os) LIKE '%linux%' THEN 'linux'
    WHEN LOWER(device_os) LIKE '%trident%' THEN 'trident'
    ELSE 'other'
  END AS device_family
FROM v05_with_time;

In [0]:
%sql
-- Verification
SELECT device_family,
  COUNT(*) AS n,
  SUM(is_fraud) AS frauds,
  ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraud
FROM v06_with_device
GROUP BY device_family
ORDER BY pct_fraud DESC;

device_family,n,frauds,pct_fraud
other,30893,3859,12.492
windows,47775,3131,6.554
ios,19782,1240,6.268
android,82,3,3.659
missing,471874,12056,2.555
macos,12573,278,2.211
trident,7440,96,1.290
linux,121,0,0.000


## 7. flag null
**Why**: for some features, the **absence of a value** is a stronger fraud
signal than the value itself. During profiling and categorical analysis. A binary flag makes this signal **explicit** and directly usable.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v07_with_flags AS
SELECT *,
  CASE WHEN distance_buyer_merchant IS NULL THEN 1 ELSE 0 END AS is_null_dist1,
  CASE WHEN distance_secondary IS NULL THEN 1 ELSE 0 END AS is_null_dist2,
  CASE WHEN buyer_region IS NULL THEN 1 ELSE 0 END AS is_null_addr1,
  CASE WHEN device_type IS NULL THEN 1 ELSE 0 END AS is_null_device_type
FROM v06_with_device;

In [0]:
%sql
--Verification
SELECT
  is_null_dist1, COUNT(*) AS n,
  ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS pct_fraud
FROM v07_with_flags
GROUP BY is_null_dist1;

is_null_dist1,n,pct_fraud
0,238269,1.996
1,352271,4.516


## Final table

**Output**: `fraud.gold.fraud_features_final`

| Metric | Value |
|---|---|
| Rows | 590,540 |
| Columns | **125** (from 102 source → +23 engineered) |
| Distinct UIDs | 217,850 |
| Fraud rate | 3.499 % |

In [0]:
%sql
CREATE OR REPLACE TABLE fraud.gold.fraud_features_final AS
SELECT * FROM v07_with_flags;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT uid) AS uids,
  SUM(is_fraud) AS frauds, ROUND(100.0*SUM(is_fraud)/COUNT(*), 3) AS pct_fraud
FROM fraud.gold.fraud_features_final;

total,uids,frauds,pct_fraude
590540,217850,20663,3.499


In [0]:
# Number of rows and columns
df_final = spark.table("fraud.gold.fraud_features_final")
n_rows = df_final.count()
n_cols = len(df_final.columns)

print(f"fraud.gold.fraud_features_final")
print(f"Rows: {n_rows:,}")
print(f"Columns: {n_cols}")

# Columns added compared to the source dataset (fraud_dataset_selected)
source_cols = set(spark.table("fraud.gold.fraud_dataset_selected").columns)
new_cols = [c for c in df_final.columns if c not in source_cols]
print(f"\n  {len(new_cols)} new columns added :")
for c in new_cols:
    print(f"    - {c}")

fraud.gold.fraud_features_final
Rows    : 590,540
Columns : 125

  40 new columns added :
    - amount
    - product_type
    - card_id
    - card_bank_1
    - card_bank_2
    - card_network
    - card_phone_country
    - card_category
    - buyer_region
    - buyer_country_code
    - distance_buyer_merchant
    - distance_secondary
    - buyer_email_domain
    - recipient_email_domain
    - device_os
    - browser
    - screen_resolution
    - d1n
    - uid
    - uid_is_known
    - uid_transaction_count
    - uid_amount_mean
    - uid_amount_std
    - uid_amount_max
    - uid_amount_sum
    - uid_d10_mean
    - uid_d15_mean
    - velocity_count_1h
    - velocity_count_24h
    - velocity_count_7d
    - velocity_amount_sum_24h
    - velocity_amount_sum_7d
    - hour_of_day
    - day_of_week
    - is_weekend
    - device_family
    - is_null_dist1
    - is_null_dist2
    - is_null_addr1
    - is_null_device_type
